# Monitor: vom Alarm zur Ampel
**Internet of Production – KI, Woche 3, Block 6** · ca. 45 min · Zweiergruppen

**Faden:** Der Service beantwortet eine Frage: *Sieht dieses Fenster mit Energiedaten wie die Vorstufe einer Blockade aus?* Ein Techniker will keinen Strom einzelner Alarme, sondern einen **Zustand**: grün, gelb, rot. In diesem Notebook entwerfen Sie die Ampel, lassen sie auf einem Replay der Druckerfarm laufen und sehen sich an, was sie an die Agenten des nächsten Blocks übergibt.

**Architektur**
```
replay_server.py  --Energy, State (jede Minute)-->  Monitor  --HTTP POST /predict-->  blockage_service.py
 (OPC UA, Daten von 2026)                              |
        ^                                              v  Gesundheitswert -> Ampel
        +--- Maintenance/Jobs/M01_RepairNeeded = True -+  + betroffene Aufträge (Orders/Onnnn)
```
Der Replay-Server spielt die Daten von 2026 ab, als wären sie live: dieselben Knoten wie im Fabrik-Server des Agentenblocks (`Factory/Machines/M01 ...`, `Maintenance/Jobs`, `Orders`), dazu der Knoten `Energy` (Wh in der letzten Minute). **Warum ein Replay?** Es sind die Daten, die die Farm liefern würde, aber reproduzierbar und schnell: Eine Blockade, die Stunden dauert, lässt sich in einer Minute zeigen.

*Voraussetzung:* Notebook 02 wurde ausgeführt (die registrierten Modelle müssen existieren). Der Service aus Notebook 03 darf weiterlaufen: Dieses Notebook nutzt denselben Port und dieselben Modelle.

In [ ]:
import sys, subprocess, time, json, asyncio, logging, warnings
sys.path.insert(0, '..')                                           # blockage_core.py and monitor_core.py lie in the folder above
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()                          # lets us use await in a notebook cell
import mlflow, mlflow.pyfunc
from IPython.display import HTML, display
logging.getLogger('mlflow').setLevel(logging.ERROR); logging.getLogger('asyncua').setLevel(logging.WARNING); warnings.filterwarnings('ignore')
from blockage_core import *
from monitor_core import Monitor, announced, offline_evaluation, COLORS

DATA_NEW = Path('../../00_Data/printfarm_2026')
STORE = Path('..').resolve()
TRACKING_URI = f"sqlite:///{(STORE / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
START, HOURS, SPEED = '2026-01-28 12:00', 12, 600                  # replay: 12 hours of 28 Jan 2026, 600 simulated minutes per real minute
SERVICE_URL = 'http://127.0.0.1:5005'
OPC_URL = 'opc.tcp://127.0.0.1:4840/freeopcua/server/'            # the endpoint of the factory server of the agent block

try:
    base = mlflow.pyfunc.load_model('models:/blockage-zrule@production').unwrap_python_model()     # the registered z-value rule
except Exception as exc:
    raise RuntimeError('No registered model found: run notebook 02 first. / Kein registriertes Modell gefunden: Führen Sie zuerst Notebook 02 aus.') from exc
print(f'z-value rule: mean {base.mean:.3f}, sd {base.sd:.4f}, alarm if z > {base.z_limit}')

## Schritt 1: Die Ampel an gespeicherten Daten entwerfen
Ein einzelner Minutenalarm (`z > Grenze`) ist unruhig: Ein kurzer Ausschlag der Energie ließe die Ampel springen. Wir glätten die Alarme zu einem **Gesundheitswert** zwischen 0 und 1: dem exponentiell gewichteten Anteil der Alarme. Eine **Halbwertszeit** von 60 Minuten heißt: Ein Alarm zählt nach einer Stunde nur noch halb. Der Wert wird durch zwei Schwellen in eine Farbe übersetzt. Nach einer Blockade und der Reparatur beginnt der Wert wieder bei 0.

Zuerst die Alarme von Drucker 0 (Maschine `M01`) für das ganze Jahr 2026, berechnet mit der registrierten Regel (dieselben Merkmale wie im Service).

In [ ]:
d0, blockages0 = load_printer(DATA_NEW, 0)                         # printer 0 = machine M01
p0 = d0.loc[d0['State'] == 'printing', 'E']                        # energy per printing minute
z0 = (p0.rolling(120).mean() - base.mean) / base.sd                # z-value of E120
alarm0 = (z0 > base.z_limit).astype(float)[z0.notna()]             # 1 = alarm in this minute
print(f'{len(alarm0)} printing minutes, {int(alarm0.sum())} minute alarms, {len(blockages0)} blockages')

**Aufgabe:** Füllen Sie die drei Lücken: den Abklingfaktor je Minute, die Aktualisierung des Gesundheitswerts um eine neue Minute und die beiden Schwellen der Ampel. Faustregel für die Schwellen: **grün unter 0,2, gelb bis 0,5, darüber rot.**

In [ ]:
HALF_LIFE = 60                                   # minutes
decay = ___                                      # factor per minute: after HALF_LIFE minutes the weight is one half

def update(health, alarm):
    """New health score from the old one and the alarm (0 or 1) of the new minute."""
    return ___                                   # old score times decay plus the new alarm times (1 - decay)

def light(health):
    if health < ___: return 'green'
    if health < ___: return 'yellow'
    return 'red'

print(update(0.0, 1), light(0.1), light(0.3), light(0.8))      # expected: 0.0115..., green yellow red

Jetzt ein Tag von Drucker 0: der Gesundheitswert samt Farbe und die Blockaden (gestrichelte Linien).

In [ ]:
day = alarm0['2026-01-28':'2026-01-28']
stops = list(blockages0[(blockages0 >= day.index[0]) & (blockages0 < pd.Timestamp('2026-01-29'))])
todo, h, hs = list(stops), 0.0, []
for t, a in day.items():
    while todo and t > todo[0]:                                    # after a blockage and repair the monitor starts again
        h = 0.0; todo.pop(0)
    h = update(h, a); hs.append(h)
hs = pd.Series(hs, index=day.index)
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.scatter(hs.index, hs.values, c=[COLORS[light(v)] for v in hs.values], s=6)
for y in (0.2, 0.5): ax.axhline(y, color='grey', lw=0.8, ls=':')
for t in stops:
    ax.axvline(t, color='black', ls='--'); ax.text(t, 1.02, 'blockage', ha='right', fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel('health score'); ax.set_title('Printer 0 (M01), 28 Jan 2026: only printing minutes are shown')
plt.tight_layout(); plt.show()

**Vergleichen Sie einen einzelnen Ausschlag mit einem anhaltenden Anstieg. Was macht die Ampel jeweils, und was ändert die Halbwertszeit?**

*Ihre Antwort:*

## Schritt 2: Halbwertszeit und Schwelle sind Entscheidungen
Dieselben Alarme, andere Einstellungen, **ganz 2026 für Drucker 0**. *Angekündigt* = die Ampel war in den 6 Stunden vor der Blockade irgendwann rot; *falsches Rot* = Rot ohne Blockade innerhalb von 8 Stunden. Nach einer Blockade beginnt der Wert wieder bei 0.

In [ ]:
rows = []
for hl in (15, 60, 180):
    dec = 0.5 ** (1 / hl)
    for red_at in (0.35, 0.5, 0.7):
        r = offline_evaluation(alarm0, blockages0, lambda h, a: h * dec + a * (1 - dec), lambda h: 'green' if h < 0.2 else ('yellow' if h < red_at else 'red'))
        rows.append({'half-life [min]': hl, 'red above': red_at, 'announced': f"{r['announced']} of {r['blockages']}", 'share': round(r['share'], 2),
                     'median lead [h]': round(r['median lead [h]'], 1), 'false red per week': round(r['false red per week'], 2)})
pd.DataFrame(rows)

**Welche Einstellung würden Sie wählen, und wer entscheidet? Nutzen Sie das Kostenverhältnis: Eine Prüfung kostet etwa 10 Minuten, eine Blockade etwa 12 Stunden.**

*Ihre Antwort:*

## Schritt 3: Replay-Server und Service starten
Beide laufen als eigene Prozesse (wie in Notebook 03). Der Replay-Server startet **angehalten** (`--hold`) und wartet, bis der Monitor sich angemeldet hat, damit keine Minute verloren geht. Der erste Start dauert etwa 20 Sekunden (der Server liest die Daten von sechs Druckern).

Wenn Sie diese Zelle ein zweites Mal ausführen, führen Sie zuerst die letzte Zelle des Notebooks (Beenden) aus, sonst sind die Ports noch belegt.

In [ ]:
from asyncua import Client
replay = subprocess.Popen([sys.executable, 'replay_server.py', '--data', str(DATA_NEW.resolve()), '--start', START, '--hours', str(HOURS), '--speed', str(SPEED), '--hold'],
                          cwd=STORE, stdout=open(STORE / 'replay.log', 'w'), stderr=subprocess.STDOUT)
service = subprocess.Popen([sys.executable, 'blockage_service.py', '--port', '5005', '--tracking-uri', TRACKING_URI],
                           cwd=STORE, stdout=open(STORE / 'service.log', 'w'), stderr=subprocess.STDOUT)

async def opc_ready():
    try:
        async with Client(url=OPC_URL, timeout=2):
            return True
    except Exception:
        return False

service_ok = opc_ok = False
for _ in range(120):
    if replay.poll() is not None or service.poll() is not None:        # a process has ended: waiting longer does not help
        break
    if not service_ok:
        try: service_ok = requests.get(f'{SERVICE_URL}/health', timeout=2).ok
        except requests.RequestException: pass
    opc_ok = opc_ok or await opc_ready()
    if service_ok and opc_ok: break
    await asyncio.sleep(1)
if not (service_ok and opc_ok):
    raise RuntimeError('Replay server or service did not start (service: %s, OPC UA: %s). Most common reason: notebook 02 has not been run, or a port is still in use. '
                       '/ Replay-Server oder Service sind nicht gestartet. Häufigste Ursache: Notebook 02 wurde nicht ausgeführt, oder ein Port ist noch belegt.\n\n--- replay.log ---\n%s\n--- service.log ---\n%s'
                       % (service_ok, opc_ok, (STORE / 'replay.log').read_text()[-800:], (STORE / 'service.log').read_text()[-800:]))
print('replay server and service are running')

Sehen Sie sich an, was der Server anbietet. Die Agenten des nächsten Blocks nutzen genau diese Knoten. **Aufgabe:** Tragen Sie den Namen des Knotens ein, der die Energie der letzten Minute enthält.

In [ ]:
async with Client(url=OPC_URL) as client:
    ns = await client.get_namespace_index('http://ostfalia.de/ipt/factory')
    factory = await client.nodes.objects.get_child([f'{ns}:Factory'])
    print('Factory:', [(await c.read_browse_name()).Name for c in await factory.get_children()])
    m01 = await factory.get_child([f'{ns}:Machines', f'{ns}:M01'])
    print('M01:', [(await c.read_browse_name()).Name for c in await m01.get_children()])
    energy = await m01.get_child([f'{ns}:___'])                     # the node with the energy of the last minute
    print('Energy of the last minute (Wh):', await energy.read_value())

## Schritt 4: Den Monitor laufen lassen
`prime` füllt das Fenster jedes Druckers mit den letzten 360 Druckminuten **vor** dem Start aus dem Datenarchiv, damit der Monitor ab der ersten Minute urteilen kann (ein Monitor, der kalt startet, braucht sechs Druckstunden). `start` meldet sich auf `Energy` und `State` jeder Maschine an und gibt das Replay frei. Der Monitor übernimmt **Ihre** Funktionen `update` und `light`.

Jede simulierte Minute jeder druckenden Maschine wird zu einem Aufruf des Service; die Ampel wird aktualisiert, und wenn sie auf Rot springt, schreibt der Monitor `RepairNeeded = True` und sucht die Aufträge, die auf dieser Maschine laufen. Das Replay von 12 Stunden dauert etwa 75 Sekunden und läuft im Hintergrund. **Achten Sie auf:** Welche Ampel wird zuerst rot, und steht ein Drucker still, während seine Ampel rot ist? Die Geschwindigkeit können Sie während des Laufs ändern. Eine Maschine, die nicht druckt (hier steht `M02` die ganze Zeit in `ready`), liefert keine neuen Hinweise: Ihre Ampel behält die letzte Farbe, ihr Gesundheitswert ändert sich nicht.

In [ ]:
monitor = await Monitor(OPC_URL, SERVICE_URL, update, light).prime(DATA_NEW, START).start()
view = display(HTML(monitor.html()), display_id=True)
END = pd.Timestamp(START) + pd.Timedelta(hours=HOURS) - pd.Timedelta(minutes=1)

async def watch(max_seconds=1800):
    """Refresh the display every second until the replay has reached its end."""
    t0 = time.time()
    while (monitor.now() or pd.Timestamp(START)) < END and time.time() - t0 < max_seconds:
        await asyncio.sleep(1)
        view.update(HTML(monitor.html()))
    view.update(HTML(monitor.html()))

watcher = asyncio.ensure_future(watch())                           # runs in the background: this cell returns at once
print('Monitor is running in the background. You can run other cells now (for example the speed change); the next cell waits for the end.')

Die Zelle oben kehrt sofort zurück, der Monitor läuft im Hintergrund weiter, und die Anzeige darüber aktualisiert sich selbst. Währenddessen können Sie andere Zellen ausführen, zum Beispiel `await monitor.set_speed(60)` in einer neuen Zelle. **Die nächste Zelle wartet, bis das Replay zu Ende ist,** und listet dann die Ereignisse auf.

In [ ]:
await watcher                                                       # waits until the replay has ended (returns at once if it is over)
for t, name, text in monitor.events:
    print(f'{t:%H:%M}  {name}  {text}')

## Schritt 5: Hat die Ampel die Blockaden angekündigt?
Der Monitor hat jede Minute protokolliert. `announced` betrachtet jeden Stillstand eines Druckers (Zustand `error`) und prüft, ob die Ampel in den 6 Stunden davor (nach der letzten Reparatur) rot war. `scored = False` heißt, dass der Monitor in diesem Fenster nicht urteilen konnte (er sammelte noch seine ersten 360 Druckminuten). **Aufgabe:** Zählen Sie die angekündigten Blockaden und die, die überhaupt beurteilt werden konnten.

In [ ]:
log = monitor.dataframe()
table = announced(log)
n_announced = ___                                  # number of stops with a red light before them
n_scored = ___                                     # number of stops that the monitor could judge at all
print(f'{n_announced} of {len(table)} stops announced by a red light; {n_scored} of {len(table)} could be judged at all')
table

In [ ]:
fig, axes = plt.subplots(len(monitor.machines), 1, figsize=(10, 1.5 * len(monitor.machines)), sharex=True)
for ax, (name, g) in zip(axes, log.groupby('machine')):
    ax.scatter(g['time'], g['health'], c=[COLORS[c] for c in g['light']], s=4)
    ax.set_ylim(0, 1.05); ax.set_ylabel(name, rotation=0, labelpad=22)
    for t in g.loc[(g['state'] == 'error') & (g['state'].shift() != 'error'), 'time']:
        ax.axvline(t, color='black', ls='--')
axes[0].set_title('Health score and light per machine (grey: warm-up, black: stopped, dashed: stop of the printer)')
plt.tight_layout(); plt.show()

**Welche Stillstände wurden nicht angekündigt, und warum? Was würden Sie am Entwurf oder im Betrieb ändern, um sie zu erreichen, und was würde es kosten?**

*Ihre Antwort:*

## Schritt 6: Was der Monitor übergibt
Das Ergebnis für den Agentenblock ist nicht die Ampel, sondern sind die Knoten: `RepairNeeded` je Maschine und die betroffenen Aufträge. Lesen Sie die Knoten jetzt aus (das Replay ist zu Ende; der Monitor setzt `RepairNeeded` zurück, wenn ein Drucker wieder in Betrieb ist, eine noch stillstehende Maschine behält also ihren Zustand).

In [ ]:
print({name: await s['job'].read_value() for name, s in monitor.machines.items()})
print()
for t, name, text in monitor.events:
    if text.startswith('RED'):
        print(f'{t:%H:%M}  {name}  {text}')

**Die Agenten des nächsten Blocks erhalten `RepairNeeded` und die betroffenen Aufträge. Was müssen sie noch wissen, bevor sie einen Auftrag umplanen, und wer entscheidet?**

*Ihre Antwort:*

Beenden Sie alles, wenn Sie fertig sind. **Der Agentenblock startet seinen eigenen Fabrik-Server auf Port 4840, geben Sie diesen Port also frei.**

In [ ]:
await monitor.stop()
replay.terminate(); service.terminate(); replay.wait(); service.wait()
print('replay server, service and monitor stopped')